# YouTube 视频摘要

## 我的第一个前沿法学硕士项目！

欢迎来到我的第一个法学硕士项目！该项目的目标是利用大型语言模型 (LLM) 来总结 YouTube 视频。目前，它仅支持英文转录，因此您无需观看整个视频，只需阅读摘要即可！

## 重要提示
使用较长的视频进行测试时要小心，因为它们可能会消耗大量资源，并可能导致您的 ChatGPT 账单成本高昂。
如果您想降低成本，可以免费使用 Ollama。

In [ ]:
!pip install youtube-transcript-api openai

In [ ]:
# 导入

import os

import requests
from dotenv import load_dotenv
from IPython.display import Markdown, display

from openai import OpenAI
from youtube_transcript_api import YouTubeTranscriptApi
import re

# 如果运行此单元时出现错误，请转到故障排除笔记本！

In [ ]:
# 加载环境变量 variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# 检查钥匙

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
openai = OpenAI()

In [ ]:
class YoutubeVideoID:
    def __init__(self, url):
        self.url = url
        self.video_id = self.extract_video_id(url)

    def extract_video_id(self, url):
        """
        Extracts the YouTube video ID from a given URL.
        Supports both regular and shortened URLs.
        """
        # 正则表达式匹配 YouTube 视频 URL 并提取视频 ID
        regex = r"(?:https?:\/\/)?(?:www\.)?(?:youtube\.com\/(?:[^\/\n\s]+\/\S+\/|\S*\?v=)|(?:youtu\.be\/))([a-zA-Z0-9_-]{11})"
        match = re.match(regex, url)
        
        if match:
            return match.group(1)
        else:
            raise ValueError("Invalid YouTube URL")

    def __str__(self):
        return f"Video ID: {self.video_id}"

In [ ]:
# 用法示例
video_url = "https://www.youtube.com/watch?v=kqaMIFEz15s"

yt_video = YoutubeVideoID(video_url)
print(yt_video)

In [ ]:
def get_transcript(video_id, language='en'):
    try:
        # 尝试获取所需语言的成绩单（默认为印度尼西亚语）
        transcript = YouTubeTranscriptApi.get_transcript(video_id, languages=[language])
        # 将所有“文本”字段连接成一个字符串
        return " ".join([item['text'] for item in transcript])
    except Exception as e:
        print(f"Error fetching transcript: {e}")
        return None


In [ ]:
# 使用视频 ID 获取文字记录
transcript_text = get_transcript(yt_video.video_id)
print(len(transcript_text))

In [ ]:
# 使用 ChatGPT 总结文本的功能
def summarize_text(text):
    try:
        system_prompts = """
        You are a helpful assistant who provides concise and accurate summaries of text. Your task is to:
        
        - Capture the key points of the content.
        - Keep the summary brief and easy to understand.
        - Avoid summarizing overly lengthy texts or breaking them into excessively short summaries.
        - Use bullet points where appropriate to enhance clarity and structure.
        """
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": system_prompts},
                {"role": "user", "content": f"Summarize the following text:\n{text}"}
            ],
            max_tokens=200
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error summarizing text: {e}")
        return None

In [ ]:
def split_text(text, chunk_size=3000):
    """
    Splits large text into smaller chunks based on the given chunk size.
    Ensures that chunks end with a full stop where possible to maintain sentence integrity.
    
    :param text: str, the text to be split
    :param chunk_size: int, maximum size of each chunk (default 3000 characters)
    :return: list of str, where each str is a chunk of text
    """
    chunks = []
    while len(text) > chunk_size:
        # 找到块大小内或块大小处的最后一个句号
        split_point = text.rfind('.', 0, chunk_size + 1)  # +1 to include the period itself if it's at chunk_size
        if split_point == -1:  # No period found within the chunk size
            split_point = chunk_size
        
        # 附加块，确保我们不会删除可能属于句子结构一部分的空格
        chunks.append(text[:split_point + 1] if split_point != chunk_size else text[:chunk_size])
        text = text[split_point + 1:] if split_point != chunk_size else text[chunk_size:]
    
    # 将剩余文本添加为​​最终块，仅在有内容时才删除
    if text:
        chunks.append(text.strip())
    
    return chunks

transcript_chunks = split_text(transcript_text)

# 现在您可以单独总结每个块
summaries = []
for chunk in transcript_chunks:
    summary = summarize_text(chunk)
    summaries.append(summary)


# 将各个单独的摘要合并为一个摘要
full_summary = " ".join(summaries)
display(Markdown(full_summary))
